<h3 style="color:#6FA8DC; font-weight:bold">06 — Curse of Dimensionality: Complete Guide</h3>

This notebook explains the **Curse of Dimensionality** from intuition → mathematics → real ML examples → effects on algorithms → solutions.

We will cover:

1. What is dimensionality?
2. What is the Curse of Dimensionality?
3. Why does it happen?
4. Geometric intuition
5. The empty-space / sparsity problem
6. Why distance becomes less meaningful
7. Real KNN example
8. Effect on K-Means and clustering
9. Effect on model complexity and overfitting
10. Computational cost
11. Real-world high-dimensional examples
12. How to solve the Curse of Dimensionality
13. Feature Selection
14. Feature Extraction / PCA
15. Regularization
16. More training data
17. Domain knowledge
18. Practical ML workflow
19. When dimensionality reduction should and should not be used
20. Final revision flow

<h3 style="color:#6FA8DC; font-weight:bold">1. What is Dimensionality?</h3>

In a dataset, **each feature can be thought of as one dimension**.

Example:

```text
Age
Salary
Experience
```

This dataset has:

```text
3 features → 3 dimensions
```

Another example:

```text
Age
Salary
Experience
Education
City
Credit Score
```

Now we have:

```text
6 features → 6 dimensions
```

In ML:

```text
Rows    → observations / samples
Columns → features / dimensions
```

So if:

```python
X.shape = (1000, 50)
```

we have:

```text
1000 observations
50 dimensions/features
```

<h3 style="color:#6FA8DC; font-weight:bold">2. What is the Curse of Dimensionality?</h3>

The **Curse of Dimensionality** refers to the problems that occur when the number of features/dimensions becomes very large.

As dimensions increase:

```text
Dimensions ↑
     ↓
Data space becomes enormous
     ↓
Data becomes sparse
     ↓
Distances become less informative
     ↓
Models may need much more data
     ↓
Computation increases
     ↓
Overfitting can increase
```

This is called a **curse** because simply adding more features does not always give us more useful information.

Sometimes:

> More features → more complexity without enough additional information.

<h3 style="color:#6FA8DC; font-weight:bold">3. The easiest intuition → a room analogy</h3>

Imagine you have:

```text
1 dimension → a line
2 dimensions → a square
3 dimensions → a cube
```

Now imagine:

```text
10 dimensions
100 dimensions
1000 dimensions
```

We cannot visually see these spaces easily, but mathematically they become extremely large.

If our data points stay roughly the same in number while the space becomes larger and larger:

```text
Huge space
    +
Few observations
    ↓
Sparse data
```

This is one of the central ideas behind the Curse of Dimensionality.

<h3 style="color:#6FA8DC; font-weight:bold">4. Real Mathematical Example → Grid Explosion</h3>

Suppose we divide each feature into only **10 possible intervals**.

For 1 feature:

```text
10 regions
```

For 2 features:

```text
10 × 10 = 100 regions
```

For 3 features:

```text
10 × 10 × 10 = 1,000 regions
```

For 10 features:

```text
10^10 = 10,000,000,000 regions
```

For 20 features:

```text
10^20 regions
```

Notice what happened.

We did not increase the number of intervals per feature.

We only increased the **number of dimensions**.

Yet the total number of possible regions exploded.

In [ ]:
dimensions = [1, 2, 3, 5, 10, 20]

for d in dimensions:
    regions = 10 ** d
    print(f"{d:>2} dimensions → {regions:,} regions")

<h5 style="color:#78B89A; font-weight:bold;">Why is this a problem?</h5>

Suppose you have only 10,000 training samples.

With 1 dimension, 10,000 samples may cover the feature space reasonably well.

But if the space has billions or trillions of possible regions, the same 10,000 samples are spread extremely thinly.

This produces **sparsity**.

<h3 style="color:#6FA8DC; font-weight:bold">5. Sparsity → the core problem</h3>

Imagine putting 100 points inside:

```text
A small 2D square
```

The points may be reasonably close together.

Now put those same 100 points inside a huge high-dimensional space.

The points become much more isolated.

```text
Low dimensions
● ● ● ●
 ● ● ●
● ● ● ●

High dimensions

●              ●

       ●

                 ●

   ●                    ●
```

This matters especially for algorithms that depend on **nearby points**.

<h3 style="color:#6FA8DC; font-weight:bold">6. Distance becomes less useful</h3>

Algorithms such as:

- KNN
- K-Means
- DBSCAN
- Nearest Neighbor search
- some recommendation systems

depend heavily on distances.

For example:

```text
KNN
↓
Find nearby training points
↓
Use their labels
```

But when dimensionality becomes very high, distances between points can become surprisingly similar.

The nearest point may not be dramatically closer than the farthest point.

This is called **distance concentration**.

<h3 style="color:#6FA8DC; font-weight:bold">7. Real Example → KNN in 2D vs 50D</h3>

Let's create a simple classification dataset.

First, KNN works in a small number of dimensions.

Then we will add many irrelevant/noisy features and observe what happens.

The important idea:

```text
Useful information
        +
Many irrelevant dimensions
        ↓
Distance calculation becomes noisy
        ↓
KNN can struggle
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=3000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_repeated=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

knn_2d = KNeighborsClassifier(n_neighbors=5)
knn_2d.fit(X_train, y_train)

pred_2d = knn_2d.predict(X_test)

print("2D accuracy:", accuracy_score(y_test, pred_2d))

<h5 style="color:#78B89A; font-weight:bold;">Visualize the 2D problem</h5>

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.5)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Classification in 2 Dimensions")
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">Now add 48 irrelevant features</h5>

The first 2 features contain useful information.

The remaining 48 features are random noise.

So:

```text
2 useful features
+
48 irrelevant features
=
50 dimensions
```

This is a very common high-dimensional ML problem.

In [ ]:
rng = np.random.default_rng(42)

noise = rng.normal(size=(X.shape[0], 48))

X_50d = np.hstack([X, noise])

X_train, X_test, y_train, y_test = train_test_split(
    X_50d, y,
    test_size=0.25,
    random_state=42
)

knn_50d = KNeighborsClassifier(n_neighbors=5)
knn_50d.fit(X_train, y_train)

pred_50d = knn_50d.predict(X_test)

print("50D accuracy:", accuracy_score(y_test, pred_50d))

The exact accuracy can vary slightly with the generated data.

The important concept is:

```text
The extra 48 features contain no useful signal,
but KNN still uses them when calculating distance.
```

So irrelevant dimensions can make the neighborhood structure less useful.

This is one practical manifestation of the Curse of Dimensionality.

<h3 style="color:#6FA8DC; font-weight:bold">8. Why does distance become concentrated?</h3>

Suppose we calculate Euclidean distance:

```text
distance = √(
    (x1-y1)² +
    (x2-y2)² +
    ...
    (xd-yd)²
)
```

As `d` increases, more terms contribute to the distance.

If many features are irrelevant/noisy:

```text
distance
=
useful contribution
+
noise contribution
+
noise contribution
+
...
```

The useful signal can become relatively small compared with the total distance.

That is why simply adding features can hurt distance-based algorithms.

<h3 style="color:#6FA8DC; font-weight:bold">9. Effect on K-Means</h3>

K-Means uses distances to cluster points.

Its basic workflow is:

```text
Choose centroids
      ↓
Calculate distance
      ↓
Assign points to nearest centroid
      ↓
Update centroids
      ↓
Repeat
```

If the distance itself becomes less informative:

```text
High dimensionality
       ↓
Distance becomes noisy
       ↓
Cluster assignment becomes harder
       ↓
Clustering quality may decrease
```

Therefore, dimensionality reduction or feature selection can sometimes help clustering.

<h3 style="color:#6FA8DC; font-weight:bold">10. Effect on Overfitting</h3>

Suppose:

```text
100 samples
2 features
```

This is relatively simple.

Now imagine:

```text
100 samples
10,000 features
```

The model has access to a huge number of possible patterns.

Some patterns may appear useful in the training data purely by chance.

This can cause:

```text
Training performance ↑
Test performance ↓
```

which is a classic sign of **overfitting**.

Important:

> High dimensionality does not automatically mean overfitting, but it can make overfitting much easier, especially when the sample size is small relative to the number of features.

<h3 style="color:#6FA8DC; font-weight:bold">11. Computational Cost</h3>

More features also mean more computation.

If:

```text
n = number of samples
d = number of features
```

Many operations depend on both `n` and `d`.

For example, distance calculations can require work across all dimensions.

So:

```text
Features ↑
    ↓
Data storage ↑
    ↓
Distance calculations ↑
    ↓
Training / prediction cost can ↑
```

This is especially important for:
- KNN
- clustering
- similarity search
- high-dimensional datasets

<h3 style="color:#6FA8DC; font-weight:bold">12. Real-World Examples of High Dimensionality</h3>

### Example 1 — Text Classification

Suppose we have:

```text
100,000 unique words
```

Using Bag-of-Words can create:

```text
100,000 features
```

But one document may contain only a few hundred words.

So the feature matrix becomes extremely sparse.

```text
Documents → rows
Words     → columns
```

This is a classic high-dimensional sparse representation.

### Example 2 — Image Data

A:

```text
100 × 100 RGB image
```

contains:

```text
100 × 100 × 3 = 30,000 values
```

Each pixel/channel can become a feature.

A dataset with thousands of images therefore lives in a very high-dimensional space.

### Example 3 — Genomics

A biological dataset may contain:

```text
thousands of genes
```

but only:

```text
hundreds of patients
```

So:

```text
features >> samples
```

This is a particularly important setting for dimensionality reduction and feature selection.

<h3 style="color:#6FA8DC; font-weight:bold">13. How do we solve the Curse of Dimensionality?</h3>

There is no single solution.

The main approaches are:

```text
                    CURSE OF DIMENSIONALITY
                              ↓
                ┌─────────────┴─────────────┐
                ↓                           ↓
          Feature Selection          Feature Extraction
                ↓                           ↓
       Keep important features          Create new
                                       lower-dimensional
                                          features
                │                           │
                └─────────────┬─────────────┘
                              ↓
                    Dimensionality Reduction
                              ↓
                ┌─────────────┼─────────────┐
                ↓             ↓             ↓
              PCA           LDA          t-SNE/UMAP

Other supporting approaches:
- Regularization
- More training data
- Domain knowledge
- Removing redundant features
- Better feature engineering
- Sparse representations

<h3 style="color:#6FA8DC; font-weight:bold">Solution 1 — Feature Selection</h3>

Feature selection means:

> Select a useful subset of the original features.

Example:

```text
100 features
     ↓
Select 20 useful features
     ↓
20 features
```

The original columns remain unchanged.

Example:

```text
Age
Salary
Experience
Height
Weight
...
```

We might discover that only:

```text
Age
Salary
Experience
```

are useful for the target.

### Types

- Filter methods
- Wrapper methods
- Embedded methods

<h5 style="color:#78B89A; font-weight:bold;">Filter methods</h5>

Features are selected using statistical relationships.

Examples:
- Correlation
- Chi-square
- ANOVA
- Mutual information

They are generally fast because they do not require training a full model for every subset.

<h5 style="color:#78B89A; font-weight:bold;">Wrapper methods</h5>

A model is repeatedly trained using different feature subsets.

Examples:
- Recursive Feature Elimination (RFE)
- Sequential Feature Selection

They can be more computationally expensive.

<h5 style="color:#78B89A; font-weight:bold;">Embedded methods</h5>

Feature selection happens during model training.

Examples:
- L1 regularization
- Tree-based feature importance

This can combine feature selection with model training.

<h3 style="color:#6FA8DC; font-weight:bold">Solution 2 — Feature Extraction</h3>

Feature extraction creates **new features** from the original features.

Example:

```text
Feature 1
Feature 2
Feature 3
Feature 4
Feature 5
      ↓
  Feature Extraction
      ↓
Component 1
Component 2
```

Unlike feature selection:

```text
Selection  → keeps original columns
Extraction → creates new representations
```

The most important classical technique is:

**PCA — Principal Component Analysis**

<h3 style="color:#6FA8DC; font-weight:bold">Solution 3 — PCA</h3>

PCA transforms many correlated features into a smaller number of new features called **principal components**.

Conceptually:

```text
Original features
      ↓
Find directions of maximum variance
      ↓
Principal Components
      ↓
Keep the most useful components
```

Example:

```text
50 features
   ↓ PCA
10 components
```

Now the model works with 10 dimensions instead of 50.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

digits = load_digits()

X = digits.data
y = digits.target

print("Original shape:", X.shape)

The digits dataset contains images represented by many pixel features.

We can reduce those features using PCA.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("Reduced shape:", X_pca.shape)

In [ ]:
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("Total variance retained:")
print(pca.explained_variance_ratio_.sum())

<h5 style="color:#78B89A; font-weight:bold;">Visualize PCA representation</h5>

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    alpha=0.6
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Digits Dataset after PCA → 2 Dimensions")
plt.show()

The 64 original pixel features have been represented using only 2 principal components.

This makes visualization possible.

However:

> PCA is not guaranteed to preserve the information that is most useful for your specific target.

PCA preserves directions of high variance, not necessarily directions with maximum predictive power.

<h3 style="color:#6FA8DC; font-weight:bold">How to choose the number of PCA components?</h3>

Instead of blindly saying:

```python
PCA(n_components=2)
```

we can choose components based on explained variance.

Example:

```text
90% explained variance
95% explained variance
99% explained variance
```

This is often more useful for preprocessing.

In [ ]:
pca_full = PCA()

X_pca_full = pca_full.fit_transform(X_scaled)

cumulative_variance = np.cumsum(
    pca_full.explained_variance_ratio_
)

plt.figure(figsize=(9, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)

plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.grid(True)
plt.show()

In [ ]:
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print("Components required for 95% variance:", n_components_95)

<h3 style="color:#6FA8DC; font-weight:bold">Solution 4 — LDA</h3>

**Linear Discriminant Analysis (LDA)** can reduce dimensionality while considering class labels.

Important difference:

```text
PCA
→ unsupervised
→ ignores target labels

LDA
→ supervised
→ uses class labels
```

LDA tries to find directions that improve class separation.

It is especially relevant for classification problems.

<h3 style="color:#6FA8DC; font-weight:bold">Solution 5 — t-SNE / UMAP</h3>

These are nonlinear dimensionality-reduction techniques.

They are particularly useful for:
- visualization
- exploring clusters
- high-dimensional embeddings

### Important distinction

t-SNE is generally used for **visualization/exploration**, not as the default production preprocessing step for a predictive model.

UMAP can also be useful for visualization and, in some workflows, as a learned representation.

These methods will be studied separately in more detail.

<h3 style="color:#6FA8DC; font-weight:bold">Solution 6 — Regularization</h3>

Regularization does not literally reduce the number of columns.

Instead, it controls model complexity.

Examples:

```text
L1 Regularization → can push some coefficients to zero
L2 Regularization → penalizes large coefficients
```

L1 can therefore perform a form of feature selection.

Example:

```text
100 features
      ↓
L1 regularization
      ↓
Some coefficients become 0
      ↓
Effective model complexity decreases
```

This is especially useful when there are many features and some may be irrelevant.

<h3 style="color:#6FA8DC; font-weight:bold">Solution 7 — More Training Data</h3>

The Curse of Dimensionality is strongly related to the relationship between:

```text
number of samples
vs
number of dimensions
```

If we increase dimensions while keeping the dataset tiny:

```text
100 samples
10,000 features
```

the problem can become severe.

Getting more useful training data can help.

But:

> More data does not make irrelevant features useful.

So increasing sample size and reducing irrelevant dimensionality are complementary strategies.

<h3 style="color:#6FA8DC; font-weight:bold">Solution 8 — Domain Knowledge</h3>

Sometimes the best dimensionality-reduction technique is simply understanding the problem.

Example:

Suppose a model has:

```text
Age
Height
Weight
BMI
Date of Birth
...
```

Domain knowledge may tell us that some variables are:
- redundant
- duplicates
- derived from other features
- irrelevant
- leakage-prone

Removing unnecessary features before modeling can be more useful than applying a complicated algorithm.

<h3 style="color:#6FA8DC; font-weight:bold">Feature Selection vs Feature Extraction</h3>

| Feature Selection | Feature Extraction |
|---|---|
| Selects original features | Creates new features |
| Original meaning retained | New representation |
| Easier to interpret | Often less interpretable |
| Example: RFE | Example: PCA |
| Can improve model simplicity | Can compress correlated information |

Example:

```text
100 features
```

Selection:

```text
100 → choose 20 original columns
```

Extraction:

```text
100 → create 20 new components
```

<h3 style="color:#6FA8DC; font-weight:bold">Which Solution Should I Choose?</h3>

```text
Many features
     ↓
Are many features irrelevant?
     ↓
   YES
     ↓
Feature Selection
     ↓
Still too many / highly correlated?
     ↓
Feature Extraction / PCA
     ↓
Need supervised separation?
     ↓
LDA
     ↓
Need visualization?
     ↓
PCA / t-SNE / UMAP
```

Also consider:

```text
Regularization
More data
Domain knowledge
Removing duplicate/redundant features
```

<h3 style="color:#6FA8DC; font-weight:bold">PCA in a Modern ML Pipeline ⭐</h3>

If PCA is used for a model, it should normally be fitted using training data only.

A modern approach is:

```text
Raw data
   ↓
Train/Test Split
   ↓
Pipeline
   ├── Scaling
   ├── PCA
   └── Model
   ↓
Prediction
```

This prevents PCA from learning information from the test set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LogisticRegression(max_iter=2000))
])

pca_pipeline.fit(X_train, y_train)

pred = pca_pipeline.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, pred))
print(
    "Components used:",
    pca_pipeline.named_steps["pca"].n_components_
)

<h3 style="color:#6FA8DC; font-weight:bold">Important: Dimensionality Reduction is not Always Good</h3>

Reducing dimensions can cause information loss.

Example:

```text
100 features
   ↓ PCA
5 components
```

You gained:
- lower computation
- less storage
- simpler representation
- potentially less overfitting

But you may lose:
- predictive information
- interpretability
- useful rare-feature signals

Therefore:

> Do not reduce dimensions just because the feature count is large.

Evaluate the effect on the actual ML objective.

<h3 style="color:#6FA8DC; font-weight:bold">Real ML Decision Example</h3>

Suppose:

```text
1,000 samples
500 features
```

Possible workflow:

```text
                 500 features
                      ↓
             Remove obvious junk
                      ↓
                 350 features
                      ↓
          Remove redundant features
                      ↓
                 200 features
                      ↓
       Feature selection / PCA if needed
                      ↓
                  30 features
                      ↓
                  Model
                      ↓
             Cross-validation
                      ↓
            Compare performance
```

The goal is not:

```text
"Get the smallest number of features."
```

The goal is:

```text
"Find a useful representation that generalizes well."
```

<h3 style="color:#6FA8DC; font-weight:bold">Common Mistakes</h3>

❌ Removing dimensions just because they are numerous.

❌ Applying PCA before train-test splitting.

❌ Assuming PCA always improves accuracy.

❌ Using t-SNE as a default production preprocessing step.

❌ Ignoring feature scaling before PCA.

❌ Keeping thousands of irrelevant features because "more data is always better."

❌ Forgetting that dimensionality problems depend on both the number of features and the number of samples.

❌ Treating feature selection and feature extraction as the same thing.

<h3 style="color:#6FA8DC; font-weight:bold">Final Complete Revision</h3>

```text
              CURSE OF DIMENSIONALITY
                        ↓
                 Features ↑↑↑
                        ↓
        ┌───────────────┼────────────────┐
        ↓               ↓                ↓
     Sparsity       Distances        Complexity
        ↓               ↓                ↓
    Less data       KNN/KMeans        Overfitting
    per region      can struggle       risk ↑
                        ↓
                 Computational cost ↑
                        ↓
                   SOLUTIONS
                        ↓
       ┌────────────────┼────────────────┐
       ↓                ↓                ↓
 Feature Selection   PCA/LDA       Regularization
       ↓                ↓                ↓
 Keep useful        Create new      Control model
 original features  representations complexity
       │                │
       └────────────────┴───────────────┐
                                        ↓
                              More training data
                              Domain knowledge
                              Remove redundancy
                              Better feature engineering
```

### One-line definition ⭐

> **The Curse of Dimensionality is the set of problems that arise as the number of dimensions/features becomes large, causing sparsity, less useful distances, higher computational cost, and potentially greater overfitting.**

### Most important distinction

```text
Feature Selection
→ keep some original features

Feature Extraction
→ create new features

Dimensionality Reduction
→ general goal of representing data with fewer dimensions
```

### Most important practical rule

> **More features are useful only when they add useful information.**